# V2 Model Evaluator

This notebook mirrors the style of the v1 evaluator while using the current v2 environment, reward logic, and split pipeline.

Important note:
- the loaded RL model is evaluated on the real v2 environment,
- and the oracle / best-single / best-double baselines are selected using the reward configuration in the config cell,
- so the utility comparison stays apples-to-apples.


In [ ]:
from pathlib import Path
import os
import sys
import textwrap
import warnings

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".mplconfig"))

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from stable_baselines3 import A2C, DQN, PPO

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "dbs" / "original_data" / "one_and_two_opt_master.csv").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not locate project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

V2_DIR = PROJECT_ROOT / "machine_learning" / "RL" / "two_opt_chain_rl_v2"
if str(V2_DIR) not in sys.path:
    sys.path.insert(0, str(V2_DIR))

from train import build_env
from evaluation import summarize_selected_frame
from rewards import (
    SUPPORTED_REWARD_TYPES,
    single_step_reward,
    incremental_step_reward,
    compute_final_correction,
)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

ALGORITHM_CLASSES = {
    "ppo": PPO,
    "a2c": A2C,
    "dqn": DQN,
}

# ------------------------------------------------------------------
# Config
# ------------------------------------------------------------------

  # "ppo", "a2c", or "dqn"
MODEL_METHOD_NAME = None   # leave as None to use "PPO agent" / "A2C agent" / "DQN agent"

EVAL_SPLIT = "test"       # "train", "validation", or "test"

# This reward definition is used to evaluate the loaded model and to
# choose the oracle / best-single / best-double baselines.

"""
#Best for 2q reduction + runtime
MODEL_PATH = V2_DIR / "models" / "optimizer_chain_ppo_two_qubit_only_rw0p1_tqw0p5_dw0p5_frw0p25.zip"
MODEL_ALGORITHM = "ppo" 
BASELINE_REWARD_TYPE = "two_qubit_only"
RUNTIME_WEIGHT = 0.1
TWO_QUBIT_WEIGHT = 0.5
DEPTH_WEIGHT = 0.5
FINAL_REWARD_WEIGHT = 0.25
"""

"""
#Best for 2q reduction + depth
MODEL_PATH = V2_DIR / "models" / "optimizer_chain_ppo_mixed_equal_rw0p2_tqw0p5_dw0p5_frw1p0.zip"
MODEL_ALGORITHM = "ppo" 
BASELINE_REWARD_TYPE = "mixed_equal"
RUNTIME_WEIGHT = 0.2
TWO_QUBIT_WEIGHT = 0.5
DEPTH_WEIGHT = 0.5
FINAL_REWARD_WEIGHT = 0.25
"""



#Best for 2q reduction + depth + runtime 
MODEL_PATH = V2_DIR / "models" / "optimizer_chain_ppo_mixed_weighted_rw0p1_tqw1p0_dw0p1_frw0p5.zip"
MODEL_ALGORITHM = "ppo" 
BASELINE_REWARD_TYPE = "mixed_weighted"
RUNTIME_WEIGHT = 0.1
TWO_QUBIT_WEIGHT = 1
DEPTH_WEIGHT = 0.1
FINAL_REWARD_WEIGHT = 0.5

SAVE_FIGURES = True
OUTPUT_DIR = V2_DIR / "model_evaluator_figures"


## 1. Load data and rebuild the v2 splits

This uses the already-saved v2 split CSVs directly.


In [ ]:
if MODEL_ALGORITHM.lower() not in ALGORITHM_CLASSES:
    raise ValueError(f"Unsupported MODEL_ALGORITHM: {MODEL_ALGORITHM}")

if BASELINE_REWARD_TYPE not in SUPPORTED_REWARD_TYPES:
    raise ValueError(
        f"Unsupported BASELINE_REWARD_TYPE: {BASELINE_REWARD_TYPE}. "
        f"Supported values: {SUPPORTED_REWARD_TYPES}"
    )

if EVAL_SPLIT not in {"train", "validation", "test"}:
    raise ValueError('EVAL_SPLIT must be one of: "train", "validation", "test".')

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model file not found: {MODEL_PATH}")

train_df = pd.read_csv(V2_DIR / "splits" / "train.csv")
validation_df = pd.read_csv(V2_DIR / "splits" / "validation.csv")
test_df = pd.read_csv(V2_DIR / "splits" / "test.csv")

optimizer_names = [
    "qiskit_standard",
    "qiskit_ai",
    "tket",
    "wisq_rules",
    "wisq_bqskit",
]
category_names = sorted(
    pd.concat(
        [
            train_df["original_category"],
            validation_df["original_category"],
            test_df["original_category"],
        ]
    ).dropna().unique()
)

def initial_metric_frame(frame):
    return pd.DataFrame(
        {
            "num_qubits": frame["initial_num_qubits"],
            "depth": frame["initial_depth"],
            "two_qubit_gates": frame["initial_two_qubit_gates"],
            "total_gates": frame["initial_total_gates"],
            "two_qubit_depth": frame["initial_two_qubit_depth"],
            "gate_density": frame["initial_gate_density"],
            "two_qubit_ratio": frame["initial_two_qubit_ratio"],
        }
    )

def final_metric_frame(frame):
    final_total = frame["final_total_gates"].replace(0, np.nan)
    final_qubits = frame["final_num_qubits"].replace(0, np.nan)
    return pd.DataFrame(
        {
            "num_qubits": frame["final_num_qubits"],
            "depth": frame["final_depth"],
            "two_qubit_gates": frame["final_two_qubit_gates"],
            "total_gates": frame["final_total_gates"],
            "two_qubit_depth": frame["final_two_qubit_depth"],
            "gate_density": frame["final_total_gates"] / final_qubits,
            "two_qubit_ratio": frame["final_two_qubit_gates"] / final_total,
        }
    ).fillna(0.0)

train_single_df = train_df[~train_df["chain_name"].str.contains("__", regex=False)].copy()
scale_source = pd.concat(
    [initial_metric_frame(train_df), final_metric_frame(train_single_df)],
    ignore_index=True,
)
metric_scale = scale_source.quantile(0.95).replace(0, 1.0).to_dict()

def add_episode_columns(frame):
    """Add the full episode reward for each chain if that chain were selected."""
    df = frame.copy()
    row_lookup = {
        (row.original_circuit, row.chain_name): row._asdict()
        for row in df.itertuples(index=False)
    }

    episode_rewards = []
    for row in df.itertuples(index=False):
        row_dict = row._asdict()

        if "__" not in row.chain_name:
            reward = float(
                single_step_reward(
                    row_dict,
                    reward_type=BASELINE_REWARD_TYPE,
                    runtime_weight=RUNTIME_WEIGHT,
                    two_qubit_weight=TWO_QUBIT_WEIGHT,
                    depth_weight=DEPTH_WEIGHT,
                )
            )
            reward += float(
                compute_final_correction(
                    row_dict,
                    reward_type=BASELINE_REWARD_TYPE,
                    final_reward_weight=FINAL_REWARD_WEIGHT,
                    two_qubit_weight=TWO_QUBIT_WEIGHT,
                    depth_weight=DEPTH_WEIGHT,
                )
            )
        else:
            first_optimizer = row.chain_name.split("__", 1)[0]
            first_row = row_lookup[(row.original_circuit, first_optimizer)]

            reward = float(
                single_step_reward(
                    first_row,
                    reward_type=BASELINE_REWARD_TYPE,
                    runtime_weight=RUNTIME_WEIGHT,
                    two_qubit_weight=TWO_QUBIT_WEIGHT,
                    depth_weight=DEPTH_WEIGHT,
                )
            )
            reward += float(
                incremental_step_reward(
                    row_dict,
                    first_row,
                    reward_type=BASELINE_REWARD_TYPE,
                    runtime_weight=RUNTIME_WEIGHT,
                    two_qubit_weight=TWO_QUBIT_WEIGHT,
                    depth_weight=DEPTH_WEIGHT,
                )
            )
            reward += float(
                compute_final_correction(
                    row_dict,
                    reward_type=BASELINE_REWARD_TYPE,
                    final_reward_weight=FINAL_REWARD_WEIGHT,
                    two_qubit_weight=TWO_QUBIT_WEIGHT,
                    depth_weight=DEPTH_WEIGHT,
                )
            )

        episode_rewards.append(reward)

    df["episode_reward_if_selected"] = episode_rewards
    df["episode_total_runtime_seconds"] = pd.to_numeric(
        df["total_runtime_seconds"], errors="coerce"
    )
    df["episode_runtime_score"] = pd.to_numeric(
        df["total_runtime_score"], errors="coerce"
    )
    return df

train_df = add_episode_columns(train_df)
validation_df = add_episode_columns(validation_df)
test_df = add_episode_columns(test_df)

eval_frames = {
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
}
eval_df = eval_frames[EVAL_SPLIT].copy()
model_method_name = MODEL_METHOD_NAME or f"{MODEL_ALGORITHM.upper()} agent"

config_summary = pd.DataFrame(
    {
        "model_path": [str(MODEL_PATH)],
        "model_algorithm": [MODEL_ALGORITHM.lower()],
        "evaluation_split": [EVAL_SPLIT],
        "baseline_reward_type": [BASELINE_REWARD_TYPE],
        "runtime_weight": [RUNTIME_WEIGHT],
        "two_qubit_weight": [TWO_QUBIT_WEIGHT],
        "depth_weight": [DEPTH_WEIGHT],
        "final_reward_weight": [FINAL_REWARD_WEIGHT],
        "eval_circuits": [eval_df["original_circuit"].nunique()],
    }
)

split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(train_df), len(validation_df), len(test_df)],
        "circuits": [
            train_df["original_circuit"].nunique(),
            validation_df["original_circuit"].nunique(),
            test_df["original_circuit"].nunique(),
        ],
        "chains": [
            train_df["chain_name"].nunique(),
            validation_df["chain_name"].nunique(),
            test_df["chain_name"].nunique(),
        ],
    }
)

display(split_summary)
display(config_summary)


## 2. Load the model and evaluate the RL policy

The model is stepped through the real v2 environment. The environment decides the state transition, and the selected final chain is then matched back to the saved rows.


In [ ]:
model = ALGORITHM_CLASSES[MODEL_ALGORITHM.lower()].load(str(MODEL_PATH))

def evaluate_policy(model, frame, method_name):
    env = build_env(
        frame,
        optimizer_names,
        category_names,
        metric_scale,
        shuffle_each_epoch=False,
        reward_type=BASELINE_REWARD_TYPE,
        runtime_weight=RUNTIME_WEIGHT,
        two_qubit_weight=TWO_QUBIT_WEIGHT,
        depth_weight=DEPTH_WEIGHT,
        final_reward_weight=FINAL_REWARD_WEIGHT,
    )

    selected_rows = []
    invalid_first_actions = 0
    invalid_second_actions = 0

    for circuit in sorted(frame["original_circuit"].unique()):
        obs, _ = env.reset(options={"circuit": circuit})

        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(int(action))

        if info.get("invalid_action", False) or info.get("selected_chain") is None:
            invalid_first_actions += 1
            continue

        selected_chain = info["selected_chain"]

        if not (terminated or truncated):
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(int(action))
            if info.get("invalid_action", False):
                invalid_second_actions += 1
            selected_chain = info["selected_chain"]

        row = frame[
            (frame["original_circuit"] == circuit)
            & (frame["chain_name"] == selected_chain)
        ].iloc[0].copy()

        row["method"] = method_name
        row["selected_chain"] = selected_chain
        row["invalid_action"] = bool(info.get("invalid_action", False))
        row["episode_reward"] = float(row["episode_reward_if_selected"])
        row["episode_total_runtime_seconds"] = float(row["total_runtime_seconds"])
        row["episode_runtime_score"] = float(row["total_runtime_score"])
        selected_rows.append(row)

    result = pd.DataFrame(selected_rows)
    total_circuits = max(frame["original_circuit"].nunique(), 1)
    result.attrs["invalid_first_action_rate"] = invalid_first_actions / total_circuits
    result.attrs["invalid_second_action_rate"] = invalid_second_actions / total_circuits
    return result

rl_selected = evaluate_policy(model, eval_df, method_name=model_method_name)

assert rl_selected["selected_chain"].notna().all()
assert rl_selected["selected_chain"].isin(eval_df["chain_name"].unique()).all()

evaluation_checks = pd.DataFrame(
    {
        "selected_circuits": [len(rl_selected)],
        "expected_circuits": [eval_df["original_circuit"].nunique()],
        "invalid_first_action_rate": [rl_selected.attrs.get("invalid_first_action_rate", 0.0)],
        "invalid_second_action_rate": [rl_selected.attrs.get("invalid_second_action_rate", 0.0)],
    }
)

display(evaluation_checks)
display(rl_selected.head())


## 3. Build the baselines

The best single optimizer and best double chain are chosen from the training split using the reward configuration from the config cell. The oracle is chosen per circuit on the evaluation split.


In [ ]:
def fixed_chain_rows(frame, chain_name, method_name):
    rows = frame[frame["chain_name"] == chain_name].copy()
    rows["method"] = method_name
    rows["selected_chain"] = chain_name
    rows["invalid_action"] = False
    rows["episode_reward"] = rows["episode_reward_if_selected"]
    rows["episode_total_runtime_seconds"] = rows["total_runtime_seconds"]
    rows["episode_runtime_score"] = rows["total_runtime_score"]
    return rows

def best_possible_depth_rows(frame, method_name="Best possible depth reduction"):
    rows = frame.loc[
        frame.groupby("original_circuit")["depth_reduction_pct"].idxmax()
    ].copy()
    rows["method"] = method_name
    rows["selected_chain"] = rows["chain_name"]
    rows["invalid_action"] = False
    rows["episode_reward"] = rows["episode_reward_if_selected"]
    rows["episode_total_runtime_seconds"] = rows["total_runtime_seconds"]
    rows["episode_runtime_score"] = rows["total_runtime_score"]
    return rows

def best_possible_twoq_rows(frame, method_name="Best possible two-qubit gate reduction"):
    rows = frame.loc[
        frame.groupby("original_circuit")["total_two_qubit_gate_reduction_pct"].idxmax()
    ].copy()
    rows["method"] = method_name
    rows["selected_chain"] = rows["chain_name"]
    rows["invalid_action"] = False
    rows["episode_reward"] = rows["episode_reward_if_selected"]
    rows["episode_total_runtime_seconds"] = rows["total_runtime_seconds"]
    rows["episode_runtime_score"] = rows["total_runtime_score"]
    return rows

def summarize_selected(selected_rows, method_name):
    summary = summarize_selected_frame(selected_rows)
    return {
        "method": method_name,
        "circuits": len(selected_rows),
        "avg_reward_utility": summary["utility"],
        "avg_pct_reduction": summary["avg_two_qubit_gate_reduction_pct"],
        "avg_depth_reduction": summary["avg_depth_reduction_pct"],
        "avg_runtime_seconds": summary["avg_runtime_seconds"],
        "avg_runtime_score": summary["avg_runtime_score"],
    }

single_chains = sorted(
    [name for name in train_df["chain_name"].unique() if "__" not in name]
)
double_chains = sorted(
    [name for name in train_df["chain_name"].unique() if "__" in name]
)

best_single_optimizer_for_depth_reduction = (
    train_df[train_df["chain_name"].isin(single_chains)]
    .groupby("chain_name")["depth_reduction_pct"]
    .mean()
    .sort_values(ascending=False)
    .index[0]
)

best_single_optimizer_for_two_qubit_reduction = (
    train_df[train_df["chain_name"].isin(single_chains)]
    .groupby("chain_name")["total_two_qubit_gate_reduction_pct"]
    .mean()
    .sort_values(ascending=False)
    .index[0]
)

best_chain_for_depth_reduction = (
    train_df[train_df["chain_name"].isin(double_chains)]
    .groupby("chain_name")["depth_reduction_pct"]
    .mean()
    .sort_values(ascending=False)
    .index[0]
)

best_chain_for_two_qubit_reduction = (
    train_df[train_df["chain_name"].isin(double_chains)]
    .groupby("chain_name")["total_two_qubit_gate_reduction_pct"]
    .mean()
    .sort_values(ascending=False)
    .index[0]
)

best_depth_chain_label = f"Best depth reduction chain ({best_chain_for_depth_reduction})"
best_twoq_chain_label = (
    f"Best 2Q gate reduction chain ({best_chain_for_two_qubit_reduction})"
)
best_depth_single_label = (
    f"Best depth reduction single ({best_single_optimizer_for_depth_reduction})"
)
best_twoq_single_label = (
    f"Best 2Q gate reduction single ({best_single_optimizer_for_two_qubit_reduction})"
)
best_possible_depth_label = "Best possible depth reduction"
best_possible_twoq_label = "Best possible two-qubit gate reduction"

baseline_reference = pd.DataFrame(
    [
        {"baseline": best_depth_chain_label, "chain": best_chain_for_depth_reduction},
        {"baseline": best_twoq_chain_label, "chain": best_chain_for_two_qubit_reduction},
        {"baseline": best_depth_single_label, "chain": best_single_optimizer_for_depth_reduction},
        {"baseline": best_twoq_single_label, "chain": best_single_optimizer_for_two_qubit_reduction},
        {"baseline": best_possible_depth_label, "chain": "per-circuit upper bound"},
        {"baseline": best_possible_twoq_label, "chain": "per-circuit upper bound"},
    ]
)
display(baseline_reference)

best_depth_chain_selected = fixed_chain_rows(
    eval_df,
    best_chain_for_depth_reduction,
    best_depth_chain_label,
)
best_twoq_chain_selected = fixed_chain_rows(
    eval_df,
    best_chain_for_two_qubit_reduction,
    best_twoq_chain_label,
)
best_depth_single_selected = fixed_chain_rows(
    eval_df,
    best_single_optimizer_for_depth_reduction,
    best_depth_single_label,
)
best_twoq_single_selected = fixed_chain_rows(
    eval_df,
    best_single_optimizer_for_two_qubit_reduction,
    best_twoq_single_label,
)
best_possible_depth_selected = best_possible_depth_rows(
    eval_df,
    best_possible_depth_label,
)
best_possible_twoq_selected = best_possible_twoq_rows(
    eval_df,
    best_possible_twoq_label,
)

comparison_rows = [
    summarize_selected(best_depth_chain_selected, best_depth_chain_label),
    summarize_selected(best_twoq_chain_selected, best_twoq_chain_label),
    summarize_selected(best_depth_single_selected, best_depth_single_label),
    summarize_selected(best_twoq_single_selected, best_twoq_single_label),
    summarize_selected(best_possible_depth_selected, best_possible_depth_label),
    summarize_selected(best_possible_twoq_selected, best_possible_twoq_label),
    summarize_selected(rl_selected, model_method_name),
]

comparison = pd.DataFrame(comparison_rows)
comparison = comparison.sort_values("avg_reward_utility", ascending=False).reset_index(drop=True)
display(comparison)




## 4. Selected chain counts and visual comparisons

This keeps the v1-style chain-usage figures and horizontal comparison bars, while adding a runtime-score comparison plot.


In [ ]:
def prettify_label(name, wrap_width=22):
    name = str(name)
    name = name.replace("__", " --> ")
    name = name.replace("_", " ")

    replacements = {
        "qiskit standard": "Qiskit",
        "qiskit ai": "Qiskit AI",
        "wisq rules": "WISQ Rules",
        "wisq bqskit": "WISQ BQSKit",
        "tket": "TKET",
        "best depth reduction chain": "Best Depth Reduction Chain",
        "best 2q gate reduction chain": "Best 2Q Gate Reduction Chain",
        "best depth reduction single": "Best Depth Reduction Single",
        "best 2q gate reduction single": "Best 2Q Gate Reduction Single",
        "best possible depth reduction": "Best Possible Depth Reduction",
        "best possible two-qubit gate reduction": "Best Possible Two-Qubit Gate Reduction",
        "ppo agent": "PPO Agent",
        "a2c agent": "A2C Agent",
        "dqn agent": "DQN Agent",
    }

    lowered = name.lower()
    for old, new in replacements.items():
        lowered = lowered.replace(old, new)
    name = lowered

    if wrap_width is None:
        return name
    return textwrap.fill(name, width=wrap_width)

selected_chain_counts = pd.concat(
    [
        rl_selected[["method", "selected_chain"]],
        best_depth_chain_selected[["method", "selected_chain"]],
        best_twoq_chain_selected[["method", "selected_chain"]],
        best_depth_single_selected[["method", "selected_chain"]],
        best_twoq_single_selected[["method", "selected_chain"]],
        best_possible_depth_selected[["method", "selected_chain"]],
        best_possible_twoq_selected[["method", "selected_chain"]],
    ],
    ignore_index=True,
).groupby(["method", "selected_chain"]).size().reset_index(name="circuits")

selected_chain_grid = selected_chain_counts.pivot_table(
    index="method",
    columns="selected_chain",
    values="circuits",
    fill_value=0,
    aggfunc="sum",
)

method_order = [
    model_method_name,
    best_depth_chain_label,
    best_twoq_chain_label,
    best_depth_single_label,
    best_twoq_single_label,
    best_possible_depth_label,
    best_possible_twoq_label,
]
selected_chain_grid = selected_chain_grid.loc[
    [method for method in method_order if method in selected_chain_grid.index],
    selected_chain_grid.sum(axis=0).sort_values(ascending=False).index,
]

selected_chain_grid_pretty = selected_chain_grid.copy()
selected_chain_grid_pretty.index = [
    prettify_label(name, wrap_width=24) for name in selected_chain_grid_pretty.index
]
selected_chain_grid_pretty.columns = [
    prettify_label(name, wrap_width=18) for name in selected_chain_grid_pretty.columns
]

display(selected_chain_grid_pretty)

plt.figure(figsize=(max(12, 0.6 * selected_chain_grid_pretty.shape[1]), 4.5))
sns.heatmap(
    selected_chain_grid_pretty,
    annot=True,
    fmt="d",
    cmap="Blues",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": f"Number of {EVAL_SPLIT} circuits"},
)
plt.xlabel("Selected optimizer chain")
plt.ylabel("Method")
plt.title("Selected Optimizer Chain Counts by Method")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

model_chain_counts = (
    rl_selected["selected_chain"]
    .value_counts()
    .rename_axis("selected_chain")
    .reset_index(name="times_selected")
)
model_chain_counts["display_chain"] = model_chain_counts["selected_chain"].apply(
    lambda name: prettify_label(name, wrap_width=18)
)

plt.figure(figsize=(max(10, 0.6 * len(model_chain_counts)), 4.5))
bars = plt.bar(
    model_chain_counts["display_chain"],
    model_chain_counts["times_selected"],
    color="#4C78A8",
)

for bar, value in zip(bars, model_chain_counts["times_selected"]):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{int(value)}",
        ha="center",
        va="bottom",
    )

plt.xlabel("Selected optimizer chain")
plt.ylabel(f"Number of {model_method_name} selections")
plt.title(f"{model_method_name} Chain Usage on {EVAL_SPLIT.title()} Split")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

def horizontal_bar(
    data,
    x,
    xlabel,
    title,
    color="#4C78A8",
    fmt="{:.3f}",
    filename=None,
    wrap_width=22,
):
    plot_data = data.sort_values(x).copy()
    plot_data["method_pretty"] = plot_data["method"].apply(
        lambda name: prettify_label(name, wrap_width=wrap_width)
    )

    fig_height = max(3.0, 0.55 * len(plot_data))
    fig, ax = plt.subplots(figsize=(4.8, fig_height))
    ax.grid(False)
    ax.xaxis.grid(False)
    ax.yaxis.grid(False)

    bars = ax.barh(plot_data["method_pretty"], plot_data[x], color=color)
    ax.set_title(title, fontsize=11, pad=10)
    ax.set_xlabel(xlabel, fontsize=10, labelpad=8)
    ax.tick_params(axis="x", labelsize=9)
    ax.tick_params(axis="y", labelsize=8)

    max_value = plot_data[x].max()
    min_value = plot_data[x].min()
    value_range = max_value - min_value
    if value_range == 0:
        value_range = abs(max_value) if max_value != 0 else 1
    padding = 0.03 * value_range

    for bar, value in zip(bars, plot_data[x]):
        ax.text(
            bar.get_width() + padding,
            bar.get_y() + bar.get_height() / 2,
            fmt.format(value),
            va="center",
            fontsize=8,
        )

    ax.set_xlim(
        min(0, min_value) - 0.05 * value_range,
        max_value + 0.22 * value_range,
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    fig.subplots_adjust(left=0.42, right=0.96, bottom=0.16, top=0.88)

    if SAVE_FIGURES and filename is not None:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(OUTPUT_DIR / filename, bbox_inches="tight")

    plt.show()
    plt.close(fig)

utility_plot = comparison.copy()

twoq_plot = comparison[
    ~comparison["method"].isin([best_possible_depth_label])
].copy()

depth_plot = comparison[
    ~comparison["method"].isin([best_possible_twoq_label])
].copy()

runtime_plot = comparison[
    ~comparison["method"].isin([best_possible_depth_label, best_possible_twoq_label])
].copy()

runtime_score_plot = comparison[
    ~comparison["method"].isin([best_possible_depth_label, best_possible_twoq_label])
].copy()

horizontal_bar(
    utility_plot,
    "avg_reward_utility",
    "Average utility reward",
    title="Average Reward Utility by Method",
    filename="avg_reward_utility.pdf",
)

horizontal_bar(
    twoq_plot,
    "avg_pct_reduction",
    "Average two-qubit gate reduction (%)",
    title="Average Two-Qubit Gate Reduction by Method",
    color="#59A14F",
    fmt="{:.1f}%",
    filename="avg_pct_reduction.pdf",
)

horizontal_bar(
    depth_plot,
    "avg_depth_reduction",
    "Average depth reduction (%)",
    title="Average Depth Reduction by Method",
    color="#76B7B2",
    fmt="{:.1f}%",
    filename="avg_depth_reduction.pdf",
)

horizontal_bar(
    runtime_plot,
    "avg_runtime_seconds",
    "Average runtime (s)",
    title="Average Runtime by Method",
    color="#F28E2B",
    fmt="{:.2f}",
    filename="avg_runtime_seconds.pdf",
)

horizontal_bar(
    runtime_score_plot,
    "avg_runtime_score",
    "Average runtime score",
    title="Average Runtime Score by Method",
    color="#E15759",
    fmt="{:.3f}",
    filename="avg_runtime_score.pdf",
)




## 5. Per-circuit RL vs best double chain

This mirrors the v1 per-circuit comparison figure, but uses the selected best double-chain baseline from the current reward configuration.


In [ ]:
utility_compare = (
    rl_selected.set_index("original_circuit")[["selected_chain", "episode_reward"]]
    .rename(
        columns={
            "selected_chain": "model_selected_chain",
            "episode_reward": "model_utility",
        }
    )
    .join(
        best_double_selected.set_index("original_circuit")[["episode_reward"]].rename(
            columns={"episode_reward": "best_double_utility"}
        ),
        how="inner",
    )
)

utility_compare["model_minus_best_double"] = (
    utility_compare["model_utility"] - utility_compare["best_double_utility"]
)
utility_compare = utility_compare.sort_values("model_minus_best_double")

display(utility_compare)

baseline_chain_label = prettify_label(best_double_chain, wrap_width=None)
model_label_pretty = prettify_label(model_method_name, wrap_width=None)

plt.figure(figsize=(14, 4))
plt.bar(
    utility_compare.index,
    utility_compare["model_minus_best_double"],
    color=np.where(
        utility_compare["model_minus_best_double"] >= 0,
        "#4C78A8",
        "#E45756",
    ),
)
plt.axhline(0, color="black", linewidth=0.8)
plt.xticks(rotation=90)
plt.ylabel(f"{model_label_pretty} utility - {baseline_chain_label} utility")
plt.title(
    f"{model_label_pretty} Utility Difference vs {baseline_chain_label} on Each {EVAL_SPLIT.title()} Circuit"
)
plt.tight_layout()
plt.show()
